# Data Cleaning

## 1. Load Raw Data from Tables

In [0]:
df_raw_2018 = spark.table("etl_project.etl_schema.df_raw_2018")
df_raw_2019 = spark.table("etl_project.etl_schema.df_raw_2019")
df_raw_2020 = spark.table("etl_project.etl_schema.df_raw_2020")
df_raw_2021 = spark.table("etl_project.etl_schema.df_raw_2021")

## 2. Drop Duplicates

In [0]:
df_clean_2018 = df_raw_2018.dropDuplicates(["entity_id"])
print(f"Loaded Dataset: {df_clean_2018.count()} rows, {len(df_clean_2018.columns)} columns")
df_clean_2019 = df_raw_2019.dropDuplicates(["entity_id"])
print(f"Loaded Dataset: {df_clean_2019.count()} rows, {len(df_clean_2019.columns)} columns")
df_clean_2020 = df_raw_2020.dropDuplicates(["entity_id"])
print(f"Loaded Dataset: {df_clean_2020.count()} rows, {len(df_clean_2020.columns)} columns")
df_clean_2021 = df_raw_2021.dropDuplicates(["entity_id"])
print(f"Loaded Dataset: {df_clean_2021.count()} rows, {len(df_clean_2021.columns)} columns")

## 3. Drop Null Values

In [0]:
df_clean_2018 = df_clean_2018.dropna(subset=["entity_id", "periodos_condicionamentos", "motivo", "creation_date", "position"])
print(f"Loaded Dataset: {df_clean_2018.count()} rows, {len(df_clean_2018.columns)} columns")
df_clean_2019 = df_clean_2019.dropna(subset=["entity_id", "periodos_condicionamentos", "motivo", "creation_date", "position"])
print(f"Loaded Dataset: {df_clean_2019.count()} rows, {len(df_clean_2019.columns)} columns")
df_clean_2020 = df_clean_2020.dropna(subset=["entity_id", "periodos_condicionamentos", "motivo", "creation_date", "position"])
print(f"Loaded Dataset: {df_clean_2020.count()} rows, {len(df_clean_2020.columns)} columns")
df_clean_2021 = df_clean_2021.dropna(subset=["entity_id", "periodos_condicionamentos", "motivo", "creation_date", "position"])
print(f"Loaded Dataset: {df_clean_2021.count()} rows, {len(df_clean_2021.columns)} columns")

## 4. Replace Motivo with Motivo Clean for Normalisation     

In [0]:
motivo_clean = {
    "OBRA - FAIXA DE RODAGEM":                "OBRA",
    "OBRAS NO SUBSOLO - FAIXA DE RODAGEM":    "OBRAS NO SUBSOLO",
    "OBRAS NO SUBSOLO - PASSEIO E ESTACION.": "OBRAS NO SUBSOLO",
    "BETONAGENS/CARGAS DESCARGAS":            "BETONAGENS/CARGAS E DESCARGAS",
    "LIGAÇÃO DE RAMAL - FAIXA DE RODAGEM":    "LIGAÇÃO DE RAMAL",
    "PLANTAÇÃO DE ÁRVORES":                   "PODA/PLANTAÇÃO DE ÁRVORES",
    "PLANTAÇÃO / PODA DE ÁRVORES":            "PODA/PLANTAÇÃO DE ÁRVORES",
    "PODA DE ÁRVORES":                        "PODA/PLANTAÇÃO DE ÁRVORES"
}

In [0]:
df_clean_2018 = df_clean_2018.replace(motivo_clean, subset=["motivo"])
df_clean_2019 = df_clean_2019.replace(motivo_clean, subset=["motivo"])
df_clean_2020 = df_clean_2020.replace(motivo_clean, subset=["motivo"])
df_clean_2021 = df_clean_2021.replace(motivo_clean, subset=["motivo"])

## 5. Removes space inconsistencies in the column "morada"

In [0]:
from pyspark.sql.functions import col, regexp_replace, trim

df_clean_2018 = df_clean_2018.withColumn(
    "morada",
    trim(
        regexp_replace(
            regexp_replace(col("morada"), "\u2009", " "),
            r"\s+",
            " "
        )
    )
)

df_2019 = df_clean_2019.withColumn(
    "morada",
    trim(
        regexp_replace(
            regexp_replace(col("morada"), "\u2009", " "),
            r"\s+",
            " "
        )
    )
)

df_2020 = df_clean_2020.withColumn(
    "morada",
    trim(
        regexp_replace(
            regexp_replace(col("morada"), "\u2009", " "),
            r"\s+",
            " "
        )
    )
)

df_2021 = df_clean_2021.withColumn(
    "morada",
    trim(
        regexp_replace(
            regexp_replace(col("morada"), "\u2009", " "),
            r"\s+",
            " "
        )
    )
)

## 6. Filter out the inconsistencies in format for entity_id

In [0]:
from pyspark.sql.functions import col, when

pattern_2018 = r"^EMEL\.condicionamentoTransito\.COND-2018-\d+-\d+$"
pattern_2019 = r"^EMEL\.condicionamentoTransito\.COND-2019-\d+-\d+$"
pattern_2020 = r"^EMEL\.condicionamentoTransito\.COND-2020-\d+-\d+$"
pattern_2021 = r"^EMEL\.condicionamentoTransito\.COND-2021-\d+-\d+$"


df_clean_2018 = df_clean_2018.withColumn(
    "entity_id_flag",
    when(~col("entity_id").rlike(pattern_2018), 1).otherwise(0)
)


df_clean_2019 = df_clean_2019.withColumn(
    "entity_id_flag",
    when(~col("entity_id").rlike(pattern_2019), 1).otherwise(0)
)

df_clean_2020 = df_clean_2020.withColumn(
    "entity_id_flag",
    when(~col("entity_id").rlike(pattern_2020), 1).otherwise(0)
)


df_clean_2021 = df_clean_2021.withColumn(
    "entity_id_flag",
    when(~col("entity_id").rlike(pattern_2021), 1).otherwise(0)
)

In [0]:
df_clean_2018 = df_clean_2018.filter(col("entity_id_flag") == 0)
print(f"Loaded Dataset: {df_clean_2018.count()} rows, {len(df_clean_2018.columns)} columns")
df_clean_2019 = df_clean_2019.filter(col("entity_id_flag") == 0)
print(f"Loaded Dataset: {df_clean_2019.count()} rows, {len(df_clean_2019.columns)} columns")
df_clean_2020 = df_clean_2020.filter(col("entity_id_flag") == 0)
print(f"Loaded Dataset: {df_clean_2020.count()} rows, {len(df_clean_2020.columns)} columns")
df_clean_2021 = df_clean_2021.filter(col("entity_id_flag") == 0)
print(f"Loaded Dataset: {df_clean_2021.count()} rows, {len(df_clean_2021.columns)} columns")

# Create Views/Tables from Cleaned Tables

## 1. Create Temporary Views/Tables

In [0]:
df_clean_2018.createOrReplaceTempView("df_clean_2018")
df_clean_2019.createOrReplaceTempView("df_clean_2019")
df_clean_2020.createOrReplaceTempView("df_clean_2020")
df_clean_2021.createOrReplaceTempView("df_clean_2021")

In [0]:
%sql 

SHOW VIEWS;

## 2. Create Views/Tables

In [0]:
%sql

USE CATALOG etl_project;
USE SCHEMA etl_schema;

CREATE OR REPLACE TABLE df_2018 AS
SELECT *
FROM df_clean_2018;

In [0]:
%sql

USE CATALOG etl_project;
USE SCHEMA etl_schema;

CREATE OR REPLACE TABLE df_2019 AS
SELECT *
FROM df_clean_2019;

In [0]:
%sql
USE CATALOG etl_project;
USE SCHEMA etl_schema;

CREATE OR REPLACE TABLE df_2020 AS
SELECT *
FROM df_clean_2020;

In [0]:
%sql
USE CATALOG etl_project;
USE SCHEMA etl_schema;

CREATE OR REPLACE TABLE df_2021 AS
SELECT *
FROM df_clean_2021;